# Fine-Tune Whisper on the Bisaya Speech Corpus (Kaggle Notebook)

This is a Kaggle-notebook adaptation of Hugging Face's
[Fine-Tune Whisper for Multilingual ASR](https://huggingface.co/blog/fine-tune-whisper)
tutorial, pointed at this project's own Bisaya (Cebuano) speech corpus
instead of the tutorial's original Hindi/Common Voice example. It trains
a third ASR system for this project's benchmark, alongside the Kaldi
HMM-GMM model (`train_kaldi.ipynb`) and ElevenLabs Scribe
(`evaluate_elevenlabs.ipynb`) -- see the repo's `README.md` for the rest
of that pipeline. This notebook is standalone: it doesn't feed into
`compare.ipynb` automatically.

**Differences from the original Colab tutorial**, besides the dataset:
- GPU/internet setup, Hugging Face auth, and long-run persistence all use
  Kaggle's own mechanisms instead of Colab's (Section "Kaggle Setup" below).
- The corpus has to be **uploaded to Kaggle as a Dataset first** (Section
  "Upload the Corpus to Kaggle" below) -- Kaggle notebooks can't read your
  local filesystem the way Colab can read from Google Drive or a direct
  download.
- The train/test split reuses the exact same speaker-independent split
  (by speaker, ~80/20, fixed seed 42) that `train_kaldi.ipynb` uses, so
  Whisper's held-out test speakers match Kaldi's -- a prerequisite for any
  later three-way comparison.
- Whisper has no dedicated Cebuano/Bisaya language token (~99 languages
  are supported; Bisaya isn't one). This notebook uses `"Tagalog"`
  (Whisper's closest Philippine-language code, `tl`) for the
  tokenizer/generation-config language conditioning -- a real
  approximation, not an exact match, called out again where it's used.


## Kaggle Setup

1. **Enable a GPU**: notebook Settings (right sidebar) -> Accelerator ->
   GPU T4 x2 (or P100/other, if available). Kaggle currently grants a
   weekly GPU-hour quota per account -- check your remaining quota in
   Settings before starting a long run.
2. **Enable internet access**: Settings -> Internet -> On. Required for
   `pip install`, downloading the pretrained Whisper checkpoint, and
   pushing to the Hugging Face Hub.
3. **Attach the corpus dataset**: see "Upload the Corpus to Kaggle" below
   -- do this once, then attach it via Add Input on every notebook that
   needs it.
4. **Add your Hugging Face token as a Kaggle Secret**: Add-ons -> Secrets
   -> add a secret named `HF_TOKEN` with a Hugging Face
   [write access token](https://huggingface.co/settings/tokens) as the
   value. Used in "Hugging Face Authentication" below instead of the
   interactive `notebook_login()` widget, so the notebook can run
   unattended (Save & Run All).

## Upload the Corpus to Kaggle

One-time step, done outside this notebook, before it can run:

1. Go to [kaggle.com/datasets](https://www.kaggle.com/datasets) -> **New
   Dataset**.
2. Upload every file under this project's `data/bisaya_audio/` (the
   Parquet shards) -- drag-and-drop in the browser, or use the
   [Kaggle API](https://www.kaggle.com/docs/api) from the machine that
   has the corpus locally:
   ```bash
   pip install kaggle
   # ~/.kaggle/kaggle.json holds your API credentials (Kaggle account ->
   # Settings -> Create New Token)
   kaggle datasets init -p data/bisaya_audio
   # edit the generated dataset-metadata.json: set a title/id, e.g.
   #   "id": "your-kaggle-username/bisaya-audio-corpus"
   kaggle datasets create -p data/bisaya_audio
   ```
3. In this notebook (or any Kaggle notebook that needs the corpus): **Add
   Input** (right sidebar) -> search for the dataset you just created ->
   Add. It appears under `/kaggle/input/<dataset-slug>/`.
4. Set `CORPUS_DIR` in the next cell to match wherever your Parquet files
   actually land under `/kaggle/input/`.

This corpus is not public -- keep the uploaded Kaggle Dataset **Private**
unless you have the right to publish it.

## Prepare Environment

In [ ]:
!nvidia-smi


In [ ]:
!pip install --upgrade --quiet pip
!pip install --upgrade --quiet datasets[audio] transformers accelerate evaluate jiwer tensorboard gradio


### Hugging Face Authentication

Uses the `HF_TOKEN` Kaggle Secret set up above, rather than the
interactive `notebook_login()` widget the original Colab tutorial uses --
this keeps the notebook runnable unattended via Kaggle's **Save & Run
All (Commit)**.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)


## Load Dataset

Reads every Parquet shard in `CORPUS_DIR` via 🤗 `datasets` directly --
the corpus's `audio` column is already stored in `datasets`' own Audio
struct format (`{bytes, path}`, with the feature type recorded in the
Parquet file's own schema metadata), so `load_dataset("parquet", ...)`
decodes it automatically, the same way it would for a published HF
dataset like the tutorial's Common Voice.

In [ ]:
from pathlib import Path

# Adjust to match the slug/path your attached Kaggle Dataset actually uses.
CORPUS_DIR = Path("/kaggle/input/bisaya-audio-corpus")

parquet_files = sorted(str(p) for p in CORPUS_DIR.glob("*.parquet"))
assert parquet_files, f"No Parquet files found under {CORPUS_DIR} -- check the dataset is attached (Add Input)."
print(f"Found {len(parquet_files)} Parquet shard(s)")


In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset("parquet", data_files=parquet_files, split="train")
print(raw_dataset)


### Speaker-Independent Train/Test Split

Same split `train_kaldi.ipynb` Section 8 uses -- by speaker (not
utterance), ~80/20, fixed seed 42 -- so this notebook's held-out test
speakers are the same ones Kaldi never trained on. Reused verbatim rather
than re-derived, so the split stays identical if this cell or
`train_kaldi.ipynb`'s changes independently.

In [ ]:
import pandas as pd

SPLIT_SEED = 42

def speaker_independent_split(speakers, test_fraction=0.2, seed=SPLIT_SEED):
    speakers = sorted(speakers)
    shuffled = pd.Series(speakers).sample(frac=1.0, random_state=seed)
    n_test = max(1, round(len(speakers) * test_fraction))
    test_speakers = set(shuffled.iloc[:n_test])
    train_speakers = set(shuffled.iloc[n_test:])
    assert train_speakers.isdisjoint(test_speakers)
    return train_speakers, test_speakers


all_speakers = set(raw_dataset.unique("speaker_id"))
train_speakers, test_speakers = speaker_independent_split(all_speakers)

bisaya = raw_dataset.train_test_split(test_size=0.0)  # placeholder, overwritten below
bisaya["train"] = raw_dataset.filter(lambda ex: ex["speaker_id"] in train_speakers)
bisaya["test"] = raw_dataset.filter(lambda ex: ex["speaker_id"] in test_speakers)

print(f"seed = {SPLIT_SEED}")
print(f"train: {len(train_speakers)} speakers, {len(bisaya['train'])} utterances")
print(f"test:  {len(test_speakers)} speakers, {len(bisaya['test'])} utterances")


In [ ]:
# Keep only the audio + transcript columns needed for fine-tuning --
# mirrors the tutorial's own column-pruning step for Common Voice's extra
# metadata (accent, locale, etc.).
keep_cols = {"audio", "transcript"}
bisaya = bisaya.remove_columns([c for c in bisaya["train"].column_names if c not in keep_cols])
print(bisaya)


## Prepare Feature Extractor, Tokenizer and Data

### Load WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

`language="Tagalog"` is Whisper's closest built-in Philippine-language
token (`tl`) -- there is no Cebuano/Bisaya entry in Whisper's ~99-language
list. This is an approximation the model was not specifically designed
for; treat any language-conditioning benefit from it as best-effort, not
exact.

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

MODEL_CHECKPOINT = "openai/whisper-small"
LANGUAGE = "Tagalog"  # closest Whisper-supported code to Bisaya/Cebuano -- see note above

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_CHECKPOINT)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_CHECKPOINT, language=LANGUAGE, task="transcribe")
processor = WhisperProcessor.from_pretrained(MODEL_CHECKPOINT, language=LANGUAGE, task="transcribe")


### Prepare Data

In [ ]:
print(bisaya["train"][0]["transcript"])


In [ ]:
from datasets import Audio

# Whisper expects 16kHz input; cast_column resamples lazily on first access.
bisaya = bisaya.cast_column("audio", Audio(sampling_rate=16000))


In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"] = tokenizer(batch["transcript"]).input_ids
    return batch


In [ ]:
bisaya = bisaya.map(prepare_dataset, remove_columns=bisaya.column_names["train"], num_proc=2)


## Training and Evaluation

Same 🤗 Trainer-based pipeline as the original tutorial: load a
pretrained checkpoint, define a data collator, define the WER metric,
configure and run training.

### Load a Pre-Trained Checkpoint

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(MODEL_CHECKPOINT)
model.generation_config.language = LANGUAGE.lower()
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None


### Define a Data Collator

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)


### Evaluation Metrics

In [ ]:
import evaluate

metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}


### Define the Training Configuration

`push_to_hub=True` is the main persistence strategy on Kaggle, same as
the original tutorial recommends for Colab: Kaggle notebook sessions are
also ephemeral (working-directory contents don't survive past the
session unless explicitly saved), so periodic checkpoints pushed to the
Hub protect the run against an interrupted or killed session.

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-bisaya",  # change to a repo name of your choice
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    warmup_steps=500,
    max_steps=4000,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=1000,
    eval_steps=1000,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=True,
)


In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=bisaya["train"],
    eval_dataset=bisaya["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

processor.save_pretrained(training_args.output_dir)


### Training

**Running unattended on Kaggle:** unlike Colab (which needs the browser
tab open and a JS keep-alive trick to survive idle disconnects), Kaggle
runs a notebook as a real background batch job when you click **Save
Version -> Save & Run All (Commit)** -- you can close the browser and it
keeps training. Use interactive "Edit" mode only for iterating on the
early cells; switch to a commit run for the actual training job.

Watch your GPU-hour quota -- a run that hits Kaggle's session time limit
partway through is still recoverable via `push_to_hub`'s periodic
checkpoints (re-load with `from_pretrained` on your Hub repo and resume),
but budgeting `max_steps` to fit inside one session avoids that entirely.

In [ ]:
trainer.train()


Update `kwargs` to match your run, then push the model card + final
checkpoint to the Hub. `dataset_tags`/`dataset_args` are omitted since
this corpus isn't a public Hub dataset (see "Upload the Corpus to
Kaggle" above) -- don't set them to a dataset id that doesn't exist.

In [ ]:
kwargs = {
    "dataset": "Bisaya Speech Corpus (private)",
    "language": "ceb",  # ISO 639-3 for Cebuano/Bisaya -- Hub model-card metadata only;
                         # unrelated to Whisper's own "Tagalog" language token used above
    "model_name": "Whisper Small Bisaya",  # a 'pretty' name for our model
    "finetuned_from": MODEL_CHECKPOINT,
    "tasks": "automatic-speech-recognition",
}


In [ ]:
trainer.push_to_hub(**kwargs)


## Building a Demo

`share=True` is required here (unlike a local or Colab environment) --
Kaggle's notebook viewer doesn't expose a local port back to your
browser, so Gradio needs its own public tunnel URL to be reachable at
all. The link is temporary and expires when the notebook session ends.

In [ ]:
from transformers import pipeline
import gradio as gr

pipe = pipeline(model=trainer.hub_model_id or training_args.output_dir)

def transcribe(audio):
    return pipe(audio)["text"]

iface = gr.Interface(
    fn=transcribe,
    inputs=gr.Audio(sources=["microphone", "upload"], type="filepath"),
    outputs="text",
    title="Whisper Small Bisaya",
    description="Bisaya (Cebuano) speech recognition using a fine-tuned Whisper small model.",
)

iface.launch(share=True)


## Closing Remarks

This notebook fine-tunes Whisper small on this project's Bisaya corpus
using 🤗 Datasets, Transformers, and the Hugging Face Hub, adapted for
Kaggle's environment (GPU/internet setup, Kaggle Secrets for
authentication, dataset upload, and unattended long-run training via
Save & Run All). See the original
[fine-tuning blog post](https://huggingface.co/blog/fine-tune-whisper)
for the underlying theory, and this project's own `README.md`/`CLAUDE.md`
for how this fits alongside the Kaldi and ElevenLabs systems in the wider
benchmark.